In [ ]:
#library
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart \
             chromadb transformers torch torchvision torchaudio \
             pillow opencv-python-headless numpy soundfile

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import uvicorn
from pyngrok import ngrok
import nest_asyncio
import os
import shutil
import re
import uuid
import functools
import subprocess

import torch
import numpy as np
import cv2 as cv
from PIL import Image
from transformers import pipeline, BlipProcessor, BlipForConditionalGeneration
import chromadb

In [ ]:
# --- The Ear: Whisper fine-tuned ---
@functools.lru_cache(maxsize=1)
def load_ear_model():
    print("Downloading The Ear (cortex-whisper)...")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    pipe = pipeline(
        "automatic-speech-recognition",
        model="Minh1506/cortex-whisper",
        tokenizer="Minh1506/cortex-whisper",
        device=device,
        chunk_length_s=30,
        stride_length_s=5,
        return_timestamps=True,
    )
    print(f"The Ear is ready on {device}")
    return pipe

In [ ]:
# --- The Eye: BLIP captioning ---
@functools.lru_cache(maxsize=1)
def load_eye_model():
    print("Downloading The Eye (BLIP)...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base"
    ).to(device)
    print(f"The Eye Ready on {device}")
    return processor, model, device

In [ ]:
# --- The Brain: ChromaDB ---
chroma_client = chromadb.PersistentClient(path="./cortexdb_kaggle")
Acollection = chroma_client.get_or_create_collection(name="Audio")
Vcollection = chroma_client.get_or_create_collection(name="Video")
print("ChromaDB Done")

In [ ]:
load_ear_model()
load_eye_model()

In [ ]:
# THE EAR
def _load_audio_ffmpeg(file_path: str, sr: int = 16000):
    # Using ffmpeg to make the audio file become suitable for the model (mono, 16000hz)
    cmd = [
        "ffmpeg", "-nostdin", "-threads", "0",
        "-i", file_path,
        "-f", "s16le", "-ac", "1", 
        "-acodec", "pcm_s16le",
        "-ar", str(sr), "-"
    ]
    out = subprocess.run(cmd, capture_output=True, check=True).stdout
    # instead of saving output to disk, save it to ram(stdout)
    return np.frombuffer(out, np.int16).flatten().astype(np.float32) / 32768.0
    # flatten the result from -1 to 1

def get_audio_data(file_path, unique_key):
    pipe = load_ear_model()
    audio_array = _load_audio_ffmpeg(file_path)
    result = pipe({"array": audio_array, "sampling_rate": 16000})

    audio_data = []
    for chunk in result["chunks"]:
        start_time, end_time = chunk["timestamp"]
        if end_time is None:
            end_time = start_time + 5.0
        text = chunk["text"].strip()
        if not text:
            continue

        audio_data.append({
                "start": start_time,
                "end": end_time,
                "text": "Text: " + text,
                "file_id": unique_key,
            })

    print(audio_data)
    Alist_ids, Alist_document, Alist_metadatas = [], [], []
    for n, item in enumerate(audio_data, start=1):
        Alist_ids.append(f"{unique_key}_{n}")
        Alist_document.append(item["text"])
        Alist_metadatas.append({"start": item["start"], "end": item["end"], "file_id": item["file_id"]})

    return Alist_ids, Alist_document, Alist_metadatas

In [ ]:
# THE EYE
def get_vision_data(video_path, unique_key):
    processor, model, device = load_eye_model()
    visual_data = []
    capture = cv.VideoCapture(video_path)

    try:
        fps = int(capture.get(cv.CAP_PROP_FPS))
        totalfps = int(capture.get(cv.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or totalfps <= 0:
            capture.release()
            return [], [], []
    except ValueError:
        capture.release()
        return [], [], []

    for i in range(0, totalfps, fps * 5):  # Take 1 frame for 5 second each
        capture.set(cv.CAP_PROP_POS_FRAMES, i)
        success, frame = capture.read()
        if not success:
            break
        framergb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        pil_image = Image.fromarray(framergb)
        inputs = processor(pil_image, return_tensors="pt").to(device)
        output = model.generate(**inputs, repetition_penalty=1.5)
        caption = processor.decode(output[0], skip_special_tokens=True)
        visual_data.append({
            "start": max(0, (i / fps) - 1),
            "end": min((i / fps) + 5, totalfps / fps),
            "text": "Visual: " + caption,
            "file_id": unique_key,
        })

    capture.release()

    Vlist_ids, Vlist_document, Vlist_metadatas = [], [], []
    for n, item in enumerate(visual_data, start=1):
        Vlist_ids.append(f"{unique_key}_{n}")
        Vlist_document.append(item["text"])
        Vlist_metadatas.append({"start": item["start"], "end": item["end"], "file_id": item["file_id"]})

    return Vlist_ids, Vlist_document, Vlist_metadatas

In [ ]:
#THE BRAIN

@functools.lru_cache(maxsize=8)
def process_file_cached(file_path, unique_key, filename):
    Alist_ids, Alist_document, Alist_metadatas = get_audio_data(file_path, unique_key)
    if len(Alist_ids) > 0:
        Acollection.upsert(ids=Alist_ids, documents=Alist_document, metadatas=Alist_metadatas)

    if filename.lower().endswith((".mp4", ".mov", ".avi", ".mkv")):
        Vlist_ids, Vlist_document, Vlist_metadatas = get_vision_data(file_path, unique_key)
        if len(Vlist_ids) > 0:
            Vcollection.upsert(ids=Vlist_ids, documents=Vlist_document, metadatas=Vlist_metadatas)

def Working(v_query, a_query, file_path, unique_key, filename):
    process_file_cached(file_path, unique_key, filename)

    Aresult = Acollection.query(where={"file_id": unique_key}, query_texts=[a_query], n_results=3)
    Ameta = Aresult["metadatas"][0]
    Adoc  = Aresult["documents"][0]
    Adis  = Aresult["distances"][0]

    is_video = filename.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))

    if is_video and v_query:
        Vresult = Vcollection.query(where={"file_id": unique_key}, query_texts=[v_query], n_results=3)
        Vmeta = Vresult["metadatas"][0]
        Vdoc  = Vresult["documents"][0]
        Vdis  = Vresult["distances"][0]

        merged = []
        for i, (am, ad, adis) in enumerate(zip(Ameta, Adoc, Adis)):
            A_start, A_end = am["start"], am["end"]
            Ascore = abs((1 - adis) * 100) if adis < 1 else 0
            for j, (vm, vd, vdis) in enumerate(zip(Vmeta, Vdoc, Vdis)):
                if A_start <= vm["end"] and A_end >= vm["start"]:
                    Vscore = abs((1 - vdis) * 100) if vdis < 1 else 0
                    merged.append({
                        "start": A_start, "end": A_end,
                        "text": ad + " " + vd,
                        "score": (Ascore + Vscore) / 2
                    })

        merged.sort(key=lambda x: x["score"], reverse=True)
        return merged[:3]
    else:
        results = []
        for am, ad, adis in zip(Ameta, Adoc, Adis):
            score = abs((1 - adis) * 100) if adis < 1 else 0
            results.append({"start": am["start"], "end": am["end"], "text": ad, "score": score})
        return results
        

In [ ]:
app = FastAPI(title="Cortex Backend API")

@app.get("/")
def home():
    return {
        "message": "Cortex Backend running on Kaggle",
        "models": {
            "ear": "Minh1506/cortex-whisper",
            "eye": "Salesforce/blip-image-captioning-base",
            "brain": "ChromaDB (PersistentClient)",
        }
    }


@app.post("/process_file") 
async def process_file(
    file: UploadFile = File(...),
    v_query: str = Form(default=""),
    a_query: str = Form(default=""),
    unique_key: str = Form(...),
    filename: str = Form(...)    
):
    os.makedirs("/tmp/cortex", exist_ok=True)

    file_ext = os.path.splitext(filename)[1].lower()
    temp_path = f"/tmp/cortex/upload_{uuid.uuid4().hex}{file_ext}"
    #uuid.uuid4().hex generate a random string (for 2 person upload same file will not get overlap)

    # Save file upload to /tmp
    with open(temp_path, "wb") as f:
        shutil.copyfileobj(file.file, f)


    if not (v_query or a_query):
        return JSONResponse(
            status_code=400,
            content={"status": "error", "message": "Please provide query."}
        )

    try:
        matches = Working(v_query, a_query, temp_path, unique_key, filename)
        return {
            "filename": filename,
            "status": "success",
            "query": {"visual": v_query, "audio": a_query},
            "results": matches,  # list[{start, end, text, score}]
        }
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"status": "error", "message": str(e)}
        )

In [ ]:
# NGROK & SERVER 
ngrok.set_auth_token("Token")  
public_url = ngrok.connect(8000, domain="API").public_url
print(f"\n{'='*50}")
print(f"API address: {public_url}")
print(f"Docs: {public_url}/docs")
print(f"{'='*50}\n")

import asyncio
nest_asyncio.apply() 
#allow to create another event loop in an existed event loop, since kaggle notebook already has an event loop
async def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()
await run_server()